# MedAgentBench GRPO Training

Train an open-weight LLM (Qwen 2.5 3B) with **true reinforcement learning** (GRPO)
to improve performance on MedAgentBench medical EHR tasks.

**Architecture:**
1. The model generates tool calls (GET/POST/FINISH) against a live FHIR server
2. The environment executes them and returns results
3. Reward functions grade correctness using `refsol` logic
4. GRPO computes policy gradients and updates weights via QLoRA

**Requirements:**
- GPU runtime (T4 16GB minimum, A100 recommended)
- FHIR server accessible (local via ngrok, or running in Colab)

**Estimated time:** ~2-4 hours for 400 GRPO steps on T4

## 1. Setup

In [ ]:
# Install dependencies
!pip install -q trl>=1.0.0 peft>=0.15.0 bitsandbytes>=0.45.0 accelerate>=1.0.0 datasets>=3.0.0 requests pyyaml rich
!pip install -q git+https://github.com/huggingface/transformers.git

In [ ]:
# Clone the repo
!git clone https://github.com/rushilcs/MedAgentBench.git 2>/dev/null || echo 'Already cloned'
%cd MedAgentBench

import sys
sys.path.insert(0, '.')

In [ ]:
# Check GPU
import torch
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f'GPU: {gpu.name} ({gpu.total_memory / 1e9:.1f} GB)')
else:
    print('WARNING: No GPU detected. Go to Runtime > Change runtime type > GPU')

## 2. FHIR Server Setup

The training environment needs a running FHIR server. You have two options:

**Option A: ngrok tunnel from your Mac** (recommended)
1. On your Mac: `docker run -p 8080:8080 medagentbench`
2. On your Mac: `ngrok http 8080`
3. Copy the ngrok URL (e.g. `https://abc123.ngrok.io`) and paste below

**Option B: Run FHIR server in Colab** (experimental)
- Uncomment and run the cell below

In [ ]:
# Set the FHIR server URL
# Option A: ngrok URL from your Mac
FHIR_BASE = "https://YOUR_NGROK_URL.ngrok.io/fhir/"  # <-- CHANGE THIS

# Option B: localhost (if running FHIR in Colab)
# FHIR_BASE = "http://localhost:8080/fhir/"

import os
os.environ['FHIR_API_BASE'] = FHIR_BASE

# Verify connection
import requests
try:
    r = requests.get(FHIR_BASE + 'metadata', timeout=10)
    print(f'FHIR server OK (status {r.status_code})')
except Exception as e:
    print(f'FHIR server NOT reachable: {e}')
    print('Please check your FHIR_BASE URL')

## 3. Prepare Training Data

In [ ]:
import json
from rl_training.data.prepare_dataset import tasks_to_dataset, load_benchmark_dataset

# Load benchmark tasks (for evaluation later)
with open('data/medagentbench/test_data_v2.json') as f:
    benchmark_tasks = json.load(f)
print(f'Benchmark tasks: {len(benchmark_tasks)}')

# Generate training tasks
from rl_training.data.task_generator import TaskGenerator

existing_mrns = {t['eval_MRN'] for t in benchmark_tasks if 'eval_MRN' in t}
task_gen = TaskGenerator(fhir_api_base=FHIR_BASE, seed=42, existing_mrns=existing_mrns)

# Start with fewer tasks for speed on free Colab
training_tasks = task_gen.generate_all(count_per_type=10)  # 100 tasks total
print(f'Training tasks: {len(training_tasks)}')

# Convert to HF Dataset
train_dataset = tasks_to_dataset(training_tasks, FHIR_BASE)
print(f'Dataset columns: {train_dataset.column_names}')
print(train_dataset[0]['prompt'])

## 3.5. Baseline Evaluation (Pre-Training)

Run the **unmodified** Qwen 2.5 3B on the benchmark tasks so we have a
before-training baseline to compare against after GRPO.

In [ ]:
from rl_training.agent.local_policy import LocalPolicy
from rl_training.env.medagent_env import MedAgentEnv
from rl_training.evaluation.evaluator import Evaluator

MODEL_NAME = 'Qwen/Qwen2.5-3B-Instruct'

# Load the base model (no LoRA, no fine-tuning)
baseline_policy = LocalPolicy(
    model_path=MODEL_NAME,
    base_model=None,  # no adapter — raw base model
    load_in_4bit=True,
    max_user_message_chars=8000,   # trim huge FHIR JSON from GET responses
    max_context_tokens=6144,       # avoid CUDA OOM on ~15GB GPUs
    max_new_tokens=256,
    empty_cache_each_act=True,     # reduce fragmentation on Colab
)

# Build environment
with open('data/medagentbench/funcs_v1.json') as f:
    funcs = json.load(f)
env = MedAgentEnv(fhir_api_base=FHIR_BASE, funcs=funcs, max_rounds=8)

# Run baseline evaluation on all 300 benchmark tasks
evaluator = Evaluator(env=env, benchmark_tasks=benchmark_tasks)
baseline_result = evaluator.evaluate_with_policy(baseline_policy)
print('\n=== BASELINE (Qwen 2.5 3B, no training) ===')
print(baseline_result.summary())

In [ ]:
# Save baseline results
import json as _json
baseline_dict = {
    'model': MODEL_NAME,
    'checkpoint': 'baseline (no training)',
    'total': baseline_result.total,
    'correct': baseline_result.correct,
    'success_rate': baseline_result.success_rate,
    'per_task_sr': baseline_result.per_task_sr,
    'query_sr': baseline_result.query_sr,
    'action_sr': baseline_result.action_sr,
    'invalid_action_rate': baseline_result.invalid_action_rate,
    'avg_steps': baseline_result.avg_steps,
}
!mkdir -p outputs/grpo_medagent
with open('outputs/grpo_medagent/baseline_results.json', 'w') as f:
    _json.dump(baseline_dict, f, indent=2)
print('Baseline saved to outputs/grpo_medagent/baseline_results.json')

# Free GPU memory before training
del baseline_policy
import gc; gc.collect()
torch.cuda.empty_cache()
print('GPU memory cleared.')

## 4. GRPO Training (Real RL)

This is the core training loop. The model:
1. Receives a task prompt
2. Generates tool calls (get_fhir_resource, post_fhir_resource, finish)
3. The environment executes them against the live FHIR server
4. Reward functions score correctness, efficiency, and tool usage
5. GRPO computes advantages and updates weights

G=4 means 4 different completions per prompt, compared group-relatively.

In [ ]:
import torch
from peft import LoraConfig
from transformers import BitsAndBytesConfig
from trl import GRPOConfig, GRPOTrainer

from rl_training.env.trl_env import MedAgentBenchEnv
from rl_training.env.trl_rewards import correctness_reward, efficiency_reward, tool_usage_reward

MODEL_NAME = 'Qwen/Qwen2.5-3B-Instruct'

# QLoRA config
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules='all-linear',
    task_type='CAUSAL_LM',
    lora_dropout=0.05,
)

# 4-bit quantization
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# GRPO training config
grpo_config = GRPOConfig(
    output_dir='outputs/grpo_medagent',
    max_steps=200,                      # reduce for free Colab; increase for better results
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_generations=4,                   # G=4 completions per prompt
    max_completion_length=2048,
    learning_rate=5e-6,
    bf16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    save_steps=50,
    logging_steps=1,
    log_completions=True,
    report_to='none',
)

print('Initializing GRPOTrainer...')
trainer = GRPOTrainer(
    model=MODEL_NAME,
    args=grpo_config,
    peft_config=peft_config,
    train_dataset=train_dataset,
    reward_funcs=[correctness_reward, efficiency_reward, tool_usage_reward],
    environment_factory=MedAgentBenchEnv,
    model_init_kwargs={'quantization_config': quant_config},
)
print('Trainer ready.')

In [ ]:
# Check VRAM before training
if torch.cuda.is_available():
    reserved = torch.cuda.max_memory_reserved() / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM: {reserved:.1f} / {total:.1f} GB reserved')

In [ ]:
# Train!
trainer.train()

In [ ]:
# Save the trained model
trainer.save_model('outputs/grpo_medagent/final')
print('Model saved to outputs/grpo_medagent/final')

## 5. Evaluate Trained Model

Run the trained model on the 300 benchmark tasks and compare to baseline.

In [ ]:
from rl_training.agent.local_policy import LocalPolicy
from rl_training.env.medagent_env import MedAgentEnv
from rl_training.evaluation.evaluator import Evaluator

# Load trained model
policy = LocalPolicy(
    model_path='outputs/grpo_medagent/final',
    base_model=MODEL_NAME,
    load_in_4bit=True,
    max_user_message_chars=8000,
    max_context_tokens=6144,
    max_new_tokens=256,
    empty_cache_each_act=True,
)

# Build environment for evaluation
with open('data/medagentbench/funcs_v1.json') as f:
    funcs = json.load(f)
env = MedAgentEnv(fhir_api_base=FHIR_BASE, funcs=funcs, max_rounds=8)

# Evaluate
evaluator = Evaluator(env=env, benchmark_tasks=benchmark_tasks)
result = evaluator.evaluate_with_policy(policy)
print('\n' + result.summary())

In [ ]:
# Save results and compare to baseline
import json
results_dict = {
    'model': MODEL_NAME,
    'checkpoint': 'outputs/grpo_medagent/final',
    'total': result.total,
    'correct': result.correct,
    'success_rate': result.success_rate,
    'per_task_sr': result.per_task_sr,
    'query_sr': result.query_sr,
    'action_sr': result.action_sr,
    'invalid_action_rate': result.invalid_action_rate,
    'avg_steps': result.avg_steps,
}
with open('outputs/grpo_medagent/eval_results.json', 'w') as f:
    json.dump(results_dict, f, indent=2)

# Print comparison
with open('outputs/grpo_medagent/baseline_results.json') as f:
    bl = json.load(f)

print('=' * 50)
print('BASELINE vs GRPO-TRAINED')
print('=' * 50)
print(f"{'Metric':<25} {'Baseline':>10} {'GRPO':>10} {'Delta':>10}")
print('-' * 55)
for metric in ['success_rate', 'query_sr', 'action_sr', 'invalid_action_rate', 'avg_steps']:
    b = bl.get(metric, 0)
    g = results_dict.get(metric, 0)
    if isinstance(b, (int, float)) and isinstance(g, (int, float)):
        delta = g - b
        sign = '+' if delta >= 0 else ''
        print(f"{metric:<25} {b:>10.3f} {g:>10.3f} {sign}{delta:>9.3f}")
print('=' * 50)

## 6. Download Trained Model

Download the LoRA adapter to use locally or push to HuggingFace Hub.

In [ ]:
# Option A: Download as zip
!zip -r grpo_medagent_checkpoint.zip outputs/grpo_medagent/final/

# Option B: Push to HuggingFace Hub
# from huggingface_hub import notebook_login
# notebook_login()
# trainer.push_to_hub('your-username/medagentbench-grpo-qwen3b')